In [1]:
import numpy as np
import pandas as pd

# 1. Định nghĩa dữ liệu 2 Player mẫu
players_data = {
    "player_id": ["player_A", "player_B"],
    "latitude": [16.0611, 16.0752],  # Tọa độ tại Đà Nẵng
    "longitude": [108.2272, 108.1531],
    "avg_skill": [3.2, 3.5],  # Trình độ trung bình (1.0 - 5.0)
    "time_slots": [
        ["2026-10-07T18:00:00", "2026-10-07T20:00:00"],
        ["2026-10-07T18:00:00"],
    ],
    "playing_style": [["Attack", "Balanced"], ["Attack"]],
    "reliability_score": [5.0, 4.5],
}

df_players = pd.DataFrame(players_data)
print("=== DỮ LIỆU MẪU 2 PLAYERS ===")
df_players

=== DỮ LIỆU MẪU 2 PLAYERS ===


,player_id,latitude,longitude,avg_skill,time_slots,playing_style,reliability_score
0,player_A,16.0611,108.2272,3.2,"[2026-10-07T18:00:00, 2026-10-07T20:00:00]","[Attack, Balanced]",5.0
1,player_B,16.0752,108.1531,3.5,[2026-10-07T18:00:00],[Attack],4.5


In [2]:
def calculate_haversine_distance(lat1, lon1, lat2, lon2):
  """Tính khoảng cách GPS giữa 2 điểm (km)"""
  R = 6371.0
  dlat = np.radians(lat2 - lat1)
  dlon = np.radians(lon2 - lon1)
  a = (
      np.sin(dlat / 2) ** 2
      + np.cos(np.radians(lat1))
      * np.cos(np.radians(lat2))
      * np.sin(dlon / 2) ** 2
  )
  c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
  return R * c


def compute_compatibility(player_a, player_b):
  # 1. Skill Score (30 điểm max)
  skill_diff = abs(player_a["avg_skill"] - player_b["avg_skill"])
  skill_score = max(0.0, 30.0 - (skill_diff * 10.0))

  # 2. Distance Score (25 điểm max)
  dist_km = calculate_haversine_distance(
      player_a["latitude"],
      player_a["longitude"],
      player_b["latitude"],
      player_b["longitude"],
  )
  distance_score = max(0.0, 25.0 - (dist_km * 1.5))

  # 3. Time Score (20 điểm max)
  slots_a = set(player_a["time_slots"])
  slots_b = set(player_b["time_slots"])
  common_slots = slots_a.intersection(slots_b)
  time_score = 20.0 if len(common_slots) > 0 else 0.0

  # 4. Preference & Style Score (25 điểm max)
  styles_a = set(player_a["playing_style"])
  styles_b = set(player_b["playing_style"])
  style_match = (
      len(styles_a.intersection(styles_b)) / len(styles_a)
      if len(styles_a) > 0
      else 0.5
  )
  preference_score = (style_match * 15.0) + (
      (player_b["reliability_score"] / 5.0) * 10.0
  )

  # Tổng điểm Compatibility Score
  total_score = round(
      skill_score + distance_score + time_score + preference_score, 2
  )

  return {
      "player_a": player_a["player_id"],
      "player_b": player_b["player_id"],
      "distance_km": round(dist_km, 2),
      "skill_score": round(skill_score, 2),
      "distance_score": round(distance_score, 2),
      "time_score": round(time_score, 2),
      "preference_score": round(preference_score, 2),
      "compatibility_score": total_score,
  }

In [3]:
# Lấy thông tin Player A và Player B từ DataFrame
player_a = df_players.iloc[0]
player_b = df_players.iloc[1]

# Tính toán điểm tương thích
result = compute_compatibility(player_a, player_b)

# Hiển thị kết quả dạng Bảng Pandas
df_result = pd.DataFrame([result])
print("\n=== KẾT QUẢ COMPATIBILITY SCORE PROTOTYPE ===")
df_result


=== KẾT QUẢ COMPATIBILITY SCORE PROTOTYPE ===


,player_a,player_b,distance_km,skill_score,distance_score,time_score,preference_score,compatibility_score
0,player_A,player_B,8.07,27.0,12.89,20.0,16.5,76.39
